In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import re
import os

# Define Paths
zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/MedLineGlossary.zip'
extract_folder = '/content/medline_unzipped'
xml_filename = 'mplus_topics_2026-09-23.xml'
# Unzip
print("Unzipping MedlinePlus...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

xml_path = os.path.join(extract_folder, xml_filename)

# Cleaning Function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Parse Initial XML Data
print("Parsing XML data...")
tree = ET.parse(xml_path)
root = tree.getroot()

raw_records = []
for topic in root.findall('.//health-topic'):
    term = topic.get('title', '')
    summary_node = topic.find('.//full-summary')
    raw_definition = "".join(summary_node.itertext()) if summary_node is not None else ""

    raw_records.append({
        'Medical_Term': clean_text(term),
        'Lay_Language_Definition': clean_text(raw_definition)
    })

# Convert to DataFrame and log starting count
df = pd.DataFrame(raw_records)
print(f"\n--- DATA PIPELINE COUNTS ---")
print(f"Original records extracted: {len(df)}")

# Filter Empties & Nulls
df = df[df['Lay_Language_Definition'].str.len() > 15]
print(f"After removing empty/short records: {len(df)}")

# Remove Duplicates
df = df.drop_duplicates(subset=['Medical_Term'])
print(f"After removing duplicate terms: {len(df)}")

# Chunking Logic
MAX_LENGTH = 800
final_chunks = []
chunked_record_count = 0

for _, row in df.iterrows():
    term = row['Medical_Term']
    definition = row['Lay_Language_Definition']

    # If text is too long, split it by sentences
    if len(definition) > MAX_LENGTH:
        chunked_record_count += 1
        sentences = definition.split('. ')
        current_chunk = ""
        chunk_idx = 1

        for sentence in sentences:
            if len(current_chunk) + len(sentence) < MAX_LENGTH:
                current_chunk += sentence + ". "
            else:
                # Save current chunk and start new
                final_chunks.append({
                    'Medical_Term': f"{term} (Part {chunk_idx})",
                    'Lay_Language_Definition': current_chunk.strip()
                })
                current_chunk = sentence + ". "
                chunk_idx += 1

        # any remaining text
        if current_chunk:
            final_chunks.append({
                'Medical_Term': f"{term} (Part {chunk_idx})",
                'Lay_Language_Definition': current_chunk.strip()
            })

    else:
        # no chunking needed
        final_chunks.append({'Medical_Term': term, 'Lay_Language_Definition': definition})

# Final Output Log
medline_df = pd.DataFrame(final_chunks)
print(f"Records that were too long and required chunking: {chunked_record_count}")
print(f"Final chunks/entries ready for vector store: {len(medline_df)}")
print("----------------------------\n")

print("Data cleaning complete! Your 'medline_df' variable is ready for the Embeddings script.")

Unzipping MedlinePlus...
Parsing XML data...

--- DATA PIPELINE COUNTS ---
Original records extracted: 2033
After removing empty/short records: 2029
After removing duplicate terms: 1979
Records that were too long and required chunking: 1612
Final chunks/entries ready for vector store: 7003
----------------------------

Data cleaning complete! Your 'medline_df' variable is ready for the Embeddings script.
